### Agglomerative clustering, example 1 (tips-data)

This example is overly simplified, technically using Agglomerative clustering for a dataset this simple is not that viable. Check out example 2 for a more realistic use case.

In [48]:
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [49]:
# load and test dataset
df = sns.load_dataset("tips")
df

# we are going to cluster the day-column (4 options)
# just for a simple demonstration

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [50]:
# encode the day as NUMERIC labels
# this is only used by one of the metrics => ARI
encoder_day = LabelEncoder()
y_true = encoder_day.fit_transform(df['day'])

**Process all variables (numeric and categoricals)**

In [51]:
# create helper lists for numeric and categorical variables
features_numeric = ['total_bill', 'tip', 'size']
features_categorical = ['sex', 'smoker', 'time']

# MODIFY THIS SO THAT BINARY-CATEGORIES ARE HANDLED SEPARATELY

# one-hot encode categoricals
encoder = OneHotEncoder(sparse_output=False)
encoded_cats = encoder.fit_transform(df[features_categorical])
encoded_cat_df = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(features_categorical))

# combine numeric + encoded categorical
X = pd.concat([df[features_numeric].reset_index(drop=True), encoded_cat_df.reset_index(drop=True)], axis=1)

### Values need to be scaled

In [52]:
# scale only the numeric features
scaler = StandardScaler()
X[features_numeric] = scaler.fit_transform(X[features_numeric])

**Perform clustering**

In [53]:
# amount of clusters, usually it should be less than the options you have
n_clusters = 4

clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = clusterer.fit_predict(X)

**Metrics**

In [54]:
# if you can get both these metrics either 1 or very close to 1
# is means the clustering describes the original structure extremely well
# and you can replace the original high cardinality column with the cluster values instead

sil_score = silhouette_score(X, labels)
print(f"Silhouette Score: {sil_score:.4f}")

ari_score = adjusted_rand_score(y_true, labels)
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")

Silhouette Score: 0.2115
Adjusted Rand Index (ARI): 0.1674


In [55]:
df['cluster'] = labels

In [56]:
df[['day', 'cluster']]

,day,cluster
0,Sun,3
1,Sun,7
2,Sun,7
3,Sun,6
4,Sun,3
...,...,...
239,Sat,4
240,Sat,2
241,Sat,5
242,Sat,6


In [57]:
df[['day', 'cluster']].value_counts().sort_index()

day   cluster
Thur  0           6
      1          43
      2           6
      3           1
      4           5
      7           1
Fri   1           4
      2           8
      3           1
      4           1
      5           3
      6           2
Sat   0           4
      2          20
      3          11
      4          15
      5          10
      6          17
      7          10
Sun   0           4
      2           5
      3          10
      4          16
      5           7
      6          22
      7          12
Name: count, dtype: int64